In [1]:
import sys, os
import matplotlib.pyplot as plt
import seaborn as sns
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print("Added to sys.path:/", repo_root)
from fixedincomelib import *
print("Fixed Income Library is loaded.")
sns.set_style("darkgrid")

Added to sys.path:/ c:\QuantBricker
Fixed Income Library is loaded.


### Build Cross Currency Basis Swap (Non-MTM)

In [2]:
effective_date = "2026-09-17"
termination_date = "2026-12-17"
currency_d = "EUR"
currency_f = "USD"
index_d = "EONIA-1B"
index_f = "SOFR-1B"
pay_or_receive_d = "receive"
foreign_notional = 1_000_000
fx_spot_f_per_d_0 = 1.10
basis_spread = 0.0010
accrual_period_d = "3M"
accrual_period_f = "3M"
accrual_basis_d = "ACT/360"
accrual_basis_f = "ACT/360"
payment_offset_d = "2D"
payment_offset_f = "2D"

prod_xccy = qfCreateProductCrossCurrencyBasisSwapNonMTM(
    effective_date,
    termination_date,
    currency_d,
    currency_f,
    index_d,
    index_f,
    pay_or_receive_d,
    foreign_notional,
    fx_spot_f_per_d_0,
    basis_spread,
    accrual_period_d,
    accrual_period_f,
    accrual_basis_d,
    accrual_basis_f,
    payment_offset_d,
    payment_offset_f,
)

In [3]:
qfDisplayProduct(prod_xccy)

,Name,Value
0,Product Type,PRODUCT_XCCY_BASIS_SWAP_NON_MTM
1,Notional,None
2,Currency,EUR
3,Long Or Short,LONG
4,Effective Date,2026-09-17
5,Termination Date,2026-12-17
6,Domestic Currency,EUR
7,Foreign Currency,USD
8,Domestic Index,EoniaON Actual/360
9,Foreign Index,SOFRON Actual/360


### Build a Yield Curve

In [4]:
bm_list = []

# USD index curve: SOFR
content_sofr = {
    'TARGET': 'SOFR-1B',
    'OVERNIGHT INDEX FUTURE': 'SOFR-FUTURE-3M',
    'OVERNIGHT INDEX SWAP': 'USD-SOFR-OIS'
}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_INDEX', content_sofr))

# EUR index curve: EUR-OIS
content_eonia = {
    'TARGET': 'EONIA-1B',
    'OVERNIGHT INDEX SWAP': 'EUR-OIS'
}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_INDEX', content_eonia))

# USD funding curve
content_funding_usd = {
    'TARGET': 'USD',
    'REFERENCE INDEX': 'SOFR-1B',
    'SPREAD ZERO RATE': 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD'
}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_funding_usd))

# EUR local funding curve
content_funding_eur_local = {
    'TARGET': 'EONIA-1B-FLAT',
    'REFERENCE INDEX': 'EONIA-1B',
    'SPREAD ZERO RATE': 'EONIA-1B-FLAT-OVER-EONIA-1B-ZERO-SPREAD'
}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_funding_eur_local))

# EUR xccy funding curve (under USD collateral)
content_funding_eur_xccy = {
    'TARGET': 'EUR-USD-XCCY',
    'REFERENCE INDEX': 'EONIA-1B',
    'CROSS CURRENCY BASIS SWAP NON MTM': 'EUR-USD-XCCY-NON-MTM'
}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FUNDING', content_funding_eur_xccy))

# fx component
content_fx = {
    'TARGET': 'EUR-USD',
    'FX SPOT RATE': 'EUR-USD'
}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_FX', content_fx))

# USD common build method
content_usd_common = {
    'TARGET': 'USD',
    'FUNDING PARAMETERS': 'USD-FUNDING-PARAMETERS',
    'SOLVER': 'BRENTQ'
}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_COMMON', content_usd_common))

# EUR common build method
content_eur_common = {
    'TARGET': 'EUR',
    'FUNDING PARAMETERS': 'EUR-FUNDING-PARAMETERS',
    'SOLVER': 'BRENTQ'
}
bm_list.append(qfCreateBuildMethod('YIELD_CURVE_COMMON', content_eur_common))

build_method_collection = qfCreateModelBuildMethodCollection(bm_list)

In [5]:
### ois
df_fut_sofr = pd.DataFrame([
    ['2026-03-19x2026-06-18', 96.44],
    ['2026-06-18x2026-09-17', 96.70],
    ['2026-09-17x2026-12-10', 96.85],
    ['2026-12-10x2027-03-17', 96.90]
], columns=['axis1', 'values']).set_index('axis1')
data_fut_sofr = qfCreateData1D('OVERNIGHT INDEX FUTURE', 'SOFR-FUTURE-3M', df_fut_sofr)

df_swap_sofr = pd.DataFrame([
    ['1Y', 0.0300],
    ['2Y', 0.0300],
    ['3Y', 0.0300]
], columns=['axis1', 'values']).set_index('axis1')
data_swap_sofr = qfCreateData1D('OVERNIGHT INDEX SWAP', 'USD-SOFR-OIS', df_swap_sofr)

df_swap_eonia = pd.DataFrame([
    ['6M', 0.0300],
    ['1Y', 0.0300],
    ['2Y', 0.0300],
    ['3Y', 0.0300]
], columns=['axis1', 'values']).set_index('axis1')
data_swap_eonia = qfCreateData1D('OVERNIGHT INDEX SWAP', 'EUR-OIS', df_swap_eonia)

### spread zero rate
df_szr_usd = pd.DataFrame([['1Y', 0.0],['3Y', 0.0]], columns=['axis1', 'values']).set_index('axis1')
data_szr_usd = qfCreateData1D('SPREAD ZERO RATE', 'SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD', df_szr_usd)

df_szr_eur = pd.DataFrame([
    ['6M', 0.0],
    ['1Y', 0.0],
    ['2Y', 0.0],
    ['3Y', 0.0]
], columns=['axis1', 'values']).set_index('axis1')
data_szr_eur = qfCreateData1D('SPREAD ZERO RATE', 'EONIA-1B-FLAT-OVER-EONIA-1B-ZERO-SPREAD', df_szr_eur)

### fx spot
df_fx = pd.DataFrame([['0D', 1.10]], columns=['axis1', 'values']).set_index('axis1')
data_fx = qfCreateData1D('FX SPOT RATE', 'EUR-USD', df_fx)

### xccy basis quotes
df_xccy = pd.DataFrame([
    ['6M', 0.001],
    ['1Y', 0.0015],
    ['2Y', 0.0018]
], columns=['axis1', 'values']).set_index('axis1')
data_xccy = qfCreateData1D('CROSS CURRENCY BASIS SWAP NON MTM', 'EUR-USD-XCCY-NON-MTM', df_xccy)

In [6]:
# funding parameter
df_fpt_usd = pd.DataFrame([
    ['Overnight Index Future', 'SOFR-FUTURE-3M', 'USD'],
    ['Overnight Index Swap', 'USD-SOFR-OIS', 'USD']
], columns=['DATA TYPE', 'DATA CONVENTION', 'FUNDING IDENTIFIER'])
data_fpt_usd = qfCreateDataGeneric('DATA GENERIC', 'USD-FUNDING-PARAMETERS', df_fpt_usd)

df_fpt_eur = pd.DataFrame([
    ['Overnight Index Swap', 'EUR-OIS', 'EONIA-1B-FLAT'],
    ['FX Spot Rate', 'EUR-USD', 'USD'],
    ['Cross Currency Basis Swap Non MTM', 'EUR-USD-XCCY-NON-MTM', 'USD']
], columns=['DATA TYPE', 'DATA CONVENTION', 'FUNDING IDENTIFIER'])
data_fpt_eur = qfCreateDataGeneric('DATA GENERIC', 'EUR-FUNDING-PARAMETERS', df_fpt_eur)

In [7]:
data_collection = qfCreateDataCollection([
    data_fut_sofr,
    data_swap_sofr,
    data_swap_eonia,
    data_szr_usd,
    data_szr_eur,
    data_fx,
    data_xccy,
    data_fpt_usd,
    data_fpt_eur
])

In [8]:
data_collection.display()

,Data Shape,Data Type,Data Convention
0,DATA1D,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M
1,DATA1D,OVERNIGHT INDEX SWAP,USD-SOFR-OIS
2,DATA1D,OVERNIGHT INDEX SWAP,EUR-OIS
3,DATA1D,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD
4,DATA1D,SPREAD ZERO RATE,EONIA-1B-FLAT-OVER-EONIA-1B-ZERO-SPREAD
5,DATA1D,FX SPOT RATE,EUR-USD
6,DATA1D,CROSS CURRENCY BASIS SWAP NON MTM,EUR-USD-XCCY-NON-MTM
7,DATAGENERIC,DATA GENERIC,USD-FUNDING-PARAMETERS
8,DATAGENERIC,DATA GENERIC,EUR-FUNDING-PARAMETERS


In [9]:
value_date = '2026-02-11'
yc_model = qfCreateModel(value_date, 'YIELD_CURVE', data_collection, build_method_collection)

In [10]:
yc_model.components_["EUR-USD-XCCY"].state_data_

array([[ 0.50136986,  1.01369863,  2.01062205],
       [ 0.00895851, -0.0121757 , -0.00208721]])

In [11]:
vp_type = 'FUNDING INDEX PARAMETER'
content_xccy = {
    'Funding Index' : 'EUR-USD-XCCY',
    'Currencies' : 'USD',
    'Funding Indices' : 'USD'
    }
fi_vp = qfCreateValuationParameters(vp_type, content_xccy)
vpc = qfCreateValuationParametersCollection([fi_vp])

In [12]:
vpc.display()

,Name,Value
0,TYPE,FUNDING INDEX PARAMETER
0,FUNDING INDEX,EUR-USD-XCCY
1,CURRENCIES,USD
2,FUNDING INDICES,USD
3,UNDERLYING FUNDING INDEX,
0,,
0,TYPE,ANALYTIC PARAMETER
0,ANALYTIC,


### Test Valuation

In [13]:
### test pv and cash
report = qfCreateValueReport(yc_model, prod_xccy, vpc, 'pvdetailed')
pv_base = qfCreateValueReport(yc_model, prod_xccy, vpc, 'pv')[0][1]
display(report.display())

,Currency,Type,Value
0,EUR,PV,-2580.542484
1,EUR,CASH,0.000000


In [14]:
### test par rate
par_rate = qfCreateValueReport(yc_model, prod_xccy, vpc, 'parrateorspread')
print(f'The implied basis spread is: {par_rate:.2%}.')

The implied basis spread is: 1.25%.


In [15]:
### test cf report
cf_report = qfCreateValueReport(yc_model, prod_xccy, vpc, 'cashflowsreport')
cf_report.display().T

,0,1,2,3,4,5
PRODUCT_TYPE,PRODUCT_XCCY_BASIS_SWAP_NON_MTM,PRODUCT_XCCY_BASIS_SWAP_NON_MTM,PRODUCT_XCCY_BASIS_SWAP_NON_MTM,PRODUCT_XCCY_BASIS_SWAP_NON_MTM,PRODUCT_XCCY_BASIS_SWAP_NON_MTM,PRODUCT_XCCY_BASIS_SWAP_NON_MTM
VALUATION_ENGINE_TYPE,ValuationEngineProductCrossCurrencyBasisSwapNo...,ValuationEngineProductCrossCurrencyBasisSwapNo...,ValuationEngineProductCrossCurrencyBasisSwapNo...,ValuationEngineProductCrossCurrencyBasisSwapNo...,ValuationEngineProductCrossCurrencyBasisSwapNo...,ValuationEngineProductCrossCurrencyBasisSwapNo...
LEG_ID,1,2,3,3,4,4
CASHFLOW_ID,0,0,0,1,0,1
PAY_OR_RECEIVE,1.0,-1.0,1.0,-1.0,-1.0,1.0
NOTIONAL,909090.909091,909090.909091,909090.909091,909090.909091,909090.909091,909090.909091
PAY_DATE,"December 21st, 2026","December 21st, 2026","September 17th, 2026","December 21st, 2026","September 17th, 2026","December 21st, 2026"
FORECASTED_AMOUNT,6994.441145,6868.079445,909090.909091,-909090.909091,-909090.909091,909090.909091
PV,6815.734303,6675.419314,889921.810804,-885863.783088,-890368.404158,883589.518969
DISCOUNG FACTOR,0.97445,0.971948,0.978914,0.97445,0.979405,0.971948


In [16]:
### test risk
risk = qfCreateValueReport(yc_model, prod_xccy, vpc, 'firstorderrisk')
df_risk = risk.display()
df_risk.VALUES = df_risk.VALUES.round(2)
display(df_risk)

,DATA_TYPE,DATA_CONVENTION,AXIS1,AXIS2,MARKET_QUOTE,UNIT,VALUES
0,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,1Y,,0.0,0.0001,-0.67
1,SPREAD ZERO RATE,SOFR-1B-FLAT-OVER-SOFR-1B-ZERO-SPREAD,3Y,,0.0,0.0001,0.92
2,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-03-19x2026-06-18,,96.44,-0.01,-23.74
3,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-06-18x2026-09-17,,96.7,-0.01,-8.01
4,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-09-17x2026-12-10,,96.85,-0.01,-35.92
5,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,1Y,,0.03,0.0001,67.10
6,OVERNIGHT INDEX FUTURE,SOFR-FUTURE-3M,2026-12-10x2027-03-17,,96.9,-0.01,-0.00
7,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,2Y,,0.03,0.0001,0.00
8,OVERNIGHT INDEX SWAP,USD-SOFR-OIS,3Y,,0.03,0.0001,0.00
9,SPREAD ZERO RATE,EONIA-1B-FLAT-OVER-EONIA-1B-ZERO-SPREAD,6M,,0.0,0.0001,0.00
